# 5.6 Eksperimen Gabungan 204 Fitur (Clustering Tiga Polutan)

Notebook ini mengerjakan **Fase 0–3** dari rencana: normalisasi kunci daerah, merge tiga matriks 68 fitur menjadi **204 fitur**, studi interpolasi pada deret waktu Jabon, reduksi dimensi PCA (target spesifikasi: 204 / 74 / 37), dan grid clustering K-Means dengan evaluasi Silhouette.

Prinsip: tidak ada hasil yang dikarang. Target dimensi yang tidak layak dengan jumlah sampel akan dicatat sebagai *tidak layak* (bukan diada-adakan), dan otomatis aktif saat data bertambah (tinggal jalankan ulang).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

SEED = 42
BASE = "../data/downloads/ekstraksi fitur"
OUT = "../data/processed"
IMG = "../data/images/eksperimen204"
os.makedirs(OUT, exist_ok=True)
os.makedirs(IMG, exist_ok=True)
rng = np.random.RandomState(SEED)

## Fase 0 — Normalisasi kunci dan merge 204 fitur

Tiga file `ekstraksi_fitur_{co,no2,so2}.csv` digabungkan dengan **kunci nama daerah yang dinormalisasi** (strip, lower, rapatkan spasi) — bukan posisi baris. Setiap fitur diberi sufiks polutan agar maknanya jelas.

In [2]:
def norm_daerah(s):
    return " ".join(str(s).strip().lower().split())

frames, feats = {}, {}
for pol in ["co", "no2", "so2"]:
    df = pd.read_csv(f"{BASE}/ekstraksi_fitur_{pol}.csv")
    df["key"] = df["daerah"].map(norm_daerah)
    dup = df[df.duplicated("key", keep=False)].sort_values("key")
    if len(dup):
        print(f"{pol}: {dup['key'].nunique()} daerah diobservasi 2x (mahasiswa berbeda) — diagregasi mean:")
        print(dup[["id", "nama", "daerah"]].to_string(index=False))
    feat = [c for c in df.columns if c not in ("id", "nama", "daerah", "key")]
    assert len(feat) == 68, f"{pol}: {len(feat)} fitur, bukan 68!"
    g = df.groupby("key")
    agg = g[feat].mean()
    agg.insert(0, "daerah", g["daerah"].first())
    agg.insert(0, "id gabungan", g["id"].apply(lambda s: ";".join(map(str, s.tolist()))))
    agg = agg.reset_index()
    frames[pol] = agg[["key", "daerah"] + feat].rename(
        columns={c: f"{c}_{pol}" for c in feat})
    feats[pol] = feat
    print(pol, df.shape, "fitur:", len(feat))

# Cek konsistensi urutan/nama fitur antar polutan
assert feats["co"] == feats["no2"] == feats["so2"], "urutan fitur beda antar polutan!"

m = frames["co"].merge(frames["no2"], on="key", how="outer", suffixes=("", "_drop"))
m = m.merge(frames["so2"], on="key", how="outer")
daerah_cols = [c for c in m.columns if c.startswith("daerah")]
m["daerah"] = m[daerah_cols].bfill(axis=1).iloc[:, 0]
m = m.drop(columns=[c for c in daerah_cols if c != "daerah"])
feat204 = [c for c in m.columns if c.endswith(("_co", "_no2", "_so2"))]
print("kolom fitur gabungan:", len(feat204))
n_full = int(m[feat204].notna().all(axis=1).sum())
print(f"baris: {len(m)}, baris lengkap 204 fitur: {n_full}")
print("baris tak lengkap:", m.loc[~m[feat204].notna().all(axis=1), "daerah"].tolist())

full = m[m[feat204].notna().all(axis=1)].copy().reset_index(drop=True)
full[["daerah"] + feat204].to_csv(f"{OUT}/clustering_204.csv", index=False)
with open(f"{OUT}/merge_report.txt", "w") as f:
    f.write(f"baris_total={len(m)}\nbaris_lengkap={len(full)}\nfitur=204\nseed={SEED}\n")
    f.write("agregasi=mean per daerah untuk observasi ganda (Kamal,Kwanyar per polutan)\n")
    f.write("drop_karena_tak_lengkap=" + ";".join(
        m.loc[~m[feat204].notna().all(axis=1), "daerah"].tolist()) + "\n")
print("tersimpan:", f"{OUT}/clustering_204.csv", full.shape)

co: 2 daerah diobservasi 2x (mahasiswa berbeda) — diagregasi mean:
 id                       nama             daerah
 25            Laila Maghfiroh   Kamal, Bangkalan
 41 Muhammad Zaidan Nabil Rafi   Kamal, Bangkalan
 29            M. Fatihul Umam Kwanyar, Bangkalan
 34             Fahdimas akmal Kwanyar, Bangkalan
co (37, 72) fitur: 68
no2: 2 daerah diobservasi 2x (mahasiswa berbeda) — diagregasi mean:
 id                       nama             daerah
 12            Laila Maghfiroh   Kamal, Bangkalan
 25 Muhammad Zaidan Nabil Rafi   Kamal, Bangkalan
 57            M. Fatihul Umam Kwanyar, Bangkalan
 60             Fahdimas akmal Kwanyar, Bangkalan
no2 (37, 72) fitur: 68
so2: 2 daerah diobservasi 2x (mahasiswa berbeda) — diagregasi mean:
 id                       nama             daerah
 25            Laila Maghfiroh   Kamal, Bangkalan
 42 Muhammad Zaidan Nabil Rafi   Kamal, Bangkalan
 29            M. Fatihul Umam Kwanyar, Bangkalan
 34             Fahdimas akmal Kwanyar, Bangkalan
so

## Fase 1 — Studi interpolasi pada deret waktu Jabon

Tiga metode didefinisikan eksplisit agar tidak tumpang tindih (polinomial ⊂ non-linear secara matematis, dibedakan via implementasi): **linear** (garis lurus antar titik), **non-linear = cubic spline**, **polinomial derajat 3** (fit global). Dibandingkan pada deret mentah Jabon (yang masih punya NaN) dengan kriteria: (a) galat di titik valid — linear/spline = 0 karena melewati titik, polinomial ≠ 0; (b) kehalusan (rata-rata |turunan kedua|); (c) visual.

In [3]:
from scipy.interpolate import CubicSpline

RAW = "../data/downloads/jabon_pollutants_data"
rows = []
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
for ax, pol in zip(axes, ["NO2", "CO", "SO2"]):
    s = pd.read_csv(f"{RAW}/jabon_{pol}.csv", parse_dates=["date"]).set_index("date")[pol]
    y = s.to_numpy(dtype=float)
    x = np.arange(len(y))
    valid = ~np.isnan(y)
    lin = pd.Series(y, index=x).interpolate(method="linear", limit_direction="both").to_numpy()
    spl = CubicSpline(x[valid], y[valid], bc_type="natural")(x)
    coef = np.polyfit(x[valid], y[valid], 3)
    ply = np.polyval(coef, x)
    rmse_valid = {"linear": float(np.sqrt(np.mean((lin[valid] - y[valid]) ** 2))),
                  "spline": float(np.sqrt(np.mean((spl[valid] - y[valid]) ** 2))),
                  "poly3": float(np.sqrt(np.mean((ply[valid] - y[valid]) ** 2)))}
    smooth = lambda v: float(np.mean(np.abs(np.diff(v, 2))))
    rows.append({"polutan": pol, "n_missing": int((~valid).sum()),
                 **{f"rmse_valid_{k}": v for k, v in rmse_valid.items()},
                 "smooth_linear": smooth(lin), "smooth_spline": smooth(spl),
                 "smooth_poly3": smooth(ply)})
    ax.plot(x, y, ".", ms=2, alpha=0.4, label="mentah")
    ax.plot(x, lin, lw=1, label="linear")
    ax.plot(x, spl, lw=1, label="cubic spline")
    ax.plot(x, ply, lw=1, label="polinomial d=3")
    ax.set_title(f"{pol} — missing {int((~valid).sum())} dari {len(y)}")
    ax.legend(fontsize=8, ncol=4)
fig.tight_layout()
fig.savefig(f"{IMG}/fig_interpolasi_perbandingan.png", dpi=100)
interp_df = pd.DataFrame(rows)
interp_df.to_csv(f"{OUT}/interpolasi_stats.csv", index=False)
print(interp_df.to_string(index=False))

polutan  n_missing  rmse_valid_linear  rmse_valid_spline  rmse_valid_poly3  smooth_linear  smooth_spline  smooth_poly3
    NO2         48                0.0       1.899970e-22          0.000011       0.000011       0.000012  7.846112e-10
     CO         44                0.0       0.000000e+00          0.003911       0.002857       0.003002  3.828050e-07
    SO2         40                0.0       0.000000e+00          0.000270       0.000273       0.000278  2.265023e-08


## Fase 2 — Reduksi dimensi PCA (target spesifikasi: 204 / 74 / 37)

Standardisasi (Z-score) lalu PCA. Karena jumlah sampel n=34 < 74, target 74 (dan secara umum target > n) **tidak layak secara matematis** — dicatat sebagai `tidak_layak` dan otomatis aktif saat n bertambah (tinggal jalankan ulang). Sebagai gantinya dipakai dimensi berbasis variansi (95% dan 99%) yang layak.

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df = pd.read_csv(f"{OUT}/clustering_204.csv")
feat204 = [c for c in df.columns if c != "daerah"]
X = StandardScaler().fit_transform(df[feat204].to_numpy())
n, p = X.shape
print(f"matriks: {n} daerah x {p} fitur")

pca_full = PCA(random_state=SEED).fit(X)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
var_df = pd.DataFrame({"komponen": np.arange(1, len(cumvar) + 1),
                       "explained": pca_full.explained_variance_ratio_,
                       "cumulative": cumvar})
var_df.to_csv(f"{OUT}/pca_variance.csv", index=False)
print(var_df.head(10).to_string(index=False))
print(f"komponen maks = {len(cumvar)} (dibatasi n={n})")

DIMS = {}
for target in [204, 74, 37]:
    if target >= p:
        DIMS[target] = ("full_tanpa_pca", X)
        print(f"dim {target}: pakai matriks full 204 (tanpa PCA)")
    elif target > n:
        print(f"dim {target}: TIDAK LAYAK (n={n} < {target}) — dicatat, aktif saat n bertambah")
    else:
        DIMS[target] = (f"pca_{target}", PCA(n_components=target, random_state=SEED).fit_transform(X))
        print(f"dim {target}: PCA-{target}, cumvar={cumvar[target-1]:.4f}")
for name, thr in [("pca_95pct", 0.95), ("pca_99pct", 0.99)]:
    k = int(np.searchsorted(cumvar, thr) + 1)
    DIMS[name] = (name, PCA(n_components=k, random_state=SEED).fit_transform(X))
    print(f"dim {name}: {k} komponen, cumvar={cumvar[k-1]:.4f}")
    pd.DataFrame(DIMS[name][1]).to_csv(f"{OUT}/X_{name}.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(var_df["komponen"], var_df["cumulative"], marker=".", ms=3)
ax.axhline(0.95, ls="--", lw=1); ax.axhline(0.99, ls="--", lw=1)
ax.set_xlabel("jumlah komponen"); ax.set_ylabel("cumulative explained variance")
fig.tight_layout(); fig.savefig(f"{IMG}/fig_pca_cumvar.png", dpi=100)

matriks: 33 daerah x 204 fitur
 komponen  explained  cumulative
        1   0.515191    0.515191
        2   0.141305    0.656496
        3   0.077395    0.733892
        4   0.046452    0.780344
        5   0.042330    0.822674
        6   0.029280    0.851954
        7   0.020530    0.872483
        8   0.016703    0.889187
        9   0.015576    0.904762
       10   0.013252    0.918015
komponen maks = 33 (dibatasi n=33)
dim 204: pakai matriks full 204 (tanpa PCA)
dim 74: TIDAK LAYAK (n=33 < 74) — dicatat, aktif saat n bertambah
dim 37: TIDAK LAYAK (n=33 < 37) — dicatat, aktif saat n bertambah
dim pca_95pct: 14 komponen, cumvar=0.9524
dim pca_99pct: 23 komponen, cumvar=0.9902


## Fase 3 — Grid clustering K-Means dan evaluasi Silhouette

Grid: dimensi layak × k=2..10, `random_state=42, n_init=10, max_iter=300`. **Seluruh** kombinasi disimpan (tanpa cherry-picking). Label cluster = identifier, bukan tingkat polusi.

In [5]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

res = []
for dim_name, (kind, Xd) in DIMS.items():
    for k in range(2, 11):
        km = KMeans(n_clusters=k, random_state=SEED, n_init=10, max_iter=300)
        lab = km.fit_predict(Xd)
        sil = float(silhouette_score(Xd, lab)) if len(set(lab)) > 1 else float("nan")
        sizes = pd.Series(lab).value_counts().sort_index()
        res.append({"dimensi": dim_name, "jenis": kind, "k": k,
                    "silhouette": sil, "inertia": float(km.inertia_), "seed": SEED,
                    "ukuran_cluster": ";".join(map(str, sizes.tolist()))})
        if k == 2:
            pd.DataFrame({"daerah": df["daerah"].tolist(), "cluster": lab}).to_csv(
                f"{OUT}/labels_{dim_name}_k2.csv", index=False)
grid = pd.DataFrame(res)
grid.to_csv(f"{OUT}/clustering_grid_results.csv", index=False)
print(grid.to_string(index=False))
best = grid.loc[grid["silhouette"].idxmax()]
print("\nTERBAIK:", best.to_dict())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for dim_name in grid["dimensi"].unique():
    g = grid[grid["dimensi"] == dim_name]
    axes[0].plot(g["k"], g["inertia"], marker="o", ms=3, label=str(dim_name))
    axes[1].plot(g["k"], g["silhouette"], marker="o", ms=3, label=str(dim_name))
axes[0].set_title("Elbow (inertia)"); axes[1].set_title("Silhouette")
for ax in axes: ax.set_xlabel("k"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{IMG}/fig_elbow_silhouette_204.png", dpi=100)

  dimensi          jenis  k  silhouette     inertia  seed       ukuran_cluster
      204 full_tanpa_pca  2    0.753039 3165.070715    42                 32;1
      204 full_tanpa_pca  3    0.467413 2645.581265    42               31;1;1
      204 full_tanpa_pca  4    0.364617 2145.395957    42             29;1;1;2
      204 full_tanpa_pca  5    0.271463 1885.960781    42           4;1;26;1;1
      204 full_tanpa_pca  6    0.131012 1552.794197    42        2;18;1;10;1;1
      204 full_tanpa_pca  7    0.113826 1357.010474    42       1;9;15;1;4;1;2
      204 full_tanpa_pca  8    0.081503 1255.486317    42    1;10;5;1;2;1;1;12
      204 full_tanpa_pca  9    0.108191 1086.746454    42   1;3;1;2;8;1;2;1;14
      204 full_tanpa_pca 10    0.108454  985.803024    42 2;14;1;1;2;1;8;2;1;1
pca_95pct      pca_95pct  2    0.768636 2855.432539    42                 32;1
pca_95pct      pca_95pct  3    0.503160 2336.094505    42               31;1;1
pca_95pct      pca_95pct  4    0.406239 1836.273093 

## Visualisasi 1 — Peta kelengkapan data per daerah × polutan

Hijau = daerah punya 68 fitur polutan tersebut; merah = tidak ada (baris merah penuh = daerah gugur dari matriks 204). Empat daerah gugur: Manyar, Manyar Gresik, Bangkalan Kota, Kec. Bangkalan.

In [6]:
def norm2(s):
    return " ".join(str(s).strip().lower().split())

keys, lab_of = {}, {}
for pol in ["co", "no2", "so2"]:
    d = pd.read_csv(f"{BASE}/ekstraksi_fitur_{pol}.csv")
    for _, r in d.iterrows():
        k = norm2(r["daerah"])
        keys.setdefault(k, {})[pol] = True
        lab_of.setdefault(k, r["daerah"])
cov = pd.DataFrame([{"daerah": lab_of[k], "CO": int("co" in v),
                     "NO2": int("no2" in v), "SO2": int("so2" in v)}
                    for k, v in sorted(keys.items())])
fig, ax = plt.subplots(figsize=(6, 12))
ax.imshow(cov[["CO", "NO2", "SO2"]].to_numpy(), aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["CO", "NO2", "SO2"])
ax.set_yticks(range(len(cov))); ax.set_yticklabels(cov["daerah"], fontsize=7)
ax.set_title("Kelengkapan 68 fitur per daerah (hijau=ada)")
fig.tight_layout(); fig.savefig(f"{IMG}/fig_coverage_daerah.png", dpi=100)
print(f"total label: {len(cov)}, lengkap 3 polutan: {int((cov[['CO','NO2','SO2']].sum(axis=1)==3).sum())}")

total label: 37, lengkap 3 polutan: 33


## Visualisasi 2 — Silhouette per k per dimensi (bar grup)

Batang tertinggi = konfigurasi terbaik. Konsisten di semua dimensi: k=2 menang, lalu skor jatuh — pola khas satu outlier jauh (Kamal/Banyuajuh) + satu gugus besar.

In [7]:
dims = grid["dimensi"].unique().tolist()
ks = sorted(grid["k"].unique().tolist())
xw = np.arange(len(ks))
fig, ax = plt.subplots(figsize=(10, 4))
for i, d in enumerate(dims):
    g = grid[grid["dimensi"] == d].sort_values("k")
    ax.bar(xw + (i - 1) * 0.25, g["silhouette"], width=0.25, label=str(d))
ax.set_xticks(xw); ax.set_xticklabels(ks)
ax.set_xlabel("k"); ax.set_ylabel("silhouette"); ax.legend(fontsize=8)
ax.set_title("Silhouette vs k per dimensi")
fig.tight_layout(); fig.savefig(f"{IMG}/fig_silhouette_bar.png", dpi=100)
print(grid.loc[grid.groupby("dimensi")["silhouette"].idxmax()][["dimensi", "k", "silhouette"]].to_string(index=False))

  dimensi  k  silhouette
      204  2    0.753039
pca_95pct  2    0.768636
pca_99pct  2    0.756337


## Visualisasi 3 — Diagram silhouette per sampel (konfigurasi terbaik)

Setiap batang = satu daerah. Batang panjang = daerah sangat cocok di clusternya; batang pendek/negatif = daerah ambigu. Satu batang merah terpisah = outlier Kamal/Banyuajuh yang membentuk cluster sendiri.

In [8]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples

Xd_best = DIMS[best["dimensi"]][1]
lab_best = KMeans(n_clusters=int(best["k"]), random_state=SEED, n_init=10,
                  max_iter=300).fit_predict(Xd_best)
s_vals = silhouette_samples(Xd_best, lab_best)
order = np.argsort(lab_best * 10 + s_vals)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [3, 1]})
axes[0].barh(range(len(s_vals)), s_vals[order],
             color=["red" if lab_best[i] == 1 else "blue" for i in order])
axes[0].axvline(float(best["silhouette"]), color="black", ls="--", lw=1, label="rata-rata")
axes[0].set_xlabel("s(i)"); axes[0].set_title(f"Silhouette per sampel ({best['dimensi']}, k={int(best['k'])})")
axes[0].legend(fontsize=8)
sizes = pd.Series(lab_best).value_counts().sort_index()
axes[1].bar(sizes.index.astype(str), sizes.values, color=["blue", "red"])
axes[1].set_title("Ukuran cluster"); axes[1].set_xlabel("cluster")
fig.tight_layout(); fig.savefig(f"{IMG}/fig_silhouette_samples.png", dpi=100)
out_name = df.loc[lab_best == 1, "daerah"].tolist()
print("anggota cluster 1:", out_name)

anggota cluster 1: ['Kamal, Banyuajuh']


## Visualisasi 4 — Sebaran cluster di bidang PCA-2D

204 dimensi diringkas ke 2 sumbu (PC1 vs PC2) agar terlihat mata: gugus biru besar vs 1 titik merah jauh terpisah. Nama outlier diberi anotasi langsung.

In [9]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

dfm = pd.read_csv(f"{OUT}/clustering_204.csv")
Xm = StandardScaler().fit_transform(dfm[[c for c in dfm.columns if c != "daerah"]].to_numpy())
P2 = PCA(n_components=2, random_state=SEED).fit_transform(Xm)
fig, ax = plt.subplots(figsize=(9, 6))
for k, col in [(0, "blue"), (1, "red")]:
    sel = lab_best == k
    ax.scatter(P2[sel, 0], P2[sel, 1], c=col, s=40, label=f"Cluster {k} (n={sel.sum()})", alpha=0.8)
    for i in np.where(sel)[0]:
        if k == 1:
            ax.annotate(dfm.loc[i, "daerah"], (P2[i, 0], P2[i, 1]), fontsize=8)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Cluster terbaik di bidang PCA-2D")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(f"{IMG}/fig_cluster_pca2d.png", dpi=100)

## Visualisasi 5 — 10 fitur paling pembeda antar cluster

Selisih rata-rata (terstandardisasi) cluster 1 − cluster 0. Batang terpanjang = fitur yang paling membedakan outlier dari mayoritas — petunjuk domain untuk interpretasi, bukan vonis tingkat polusi.

In [10]:
c0 = Xm[lab_best == 0].mean(axis=0)
c1 = Xm[lab_best == 1].mean(axis=0)
diff = pd.Series(c1 - c0, index=[c for c in dfm.columns if c != "daerah"])
top = diff.abs().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(top.index[::-1], diff[top.index[::-1]])
ax.set_xlabel("mean(cluster 1) − mean(cluster 0), terstandardisasi")
ax.set_title("10 fitur paling pembeda")
fig.tight_layout(); fig.savefig(f"{IMG}/fig_top_pembeda.png", dpi=100)
print(diff[top.index].to_string())

average_power_no2             5.833631
abs_energy_co                 5.833631
wavelet_var_so2               5.833631
abs_energy_no2                5.833631
calc_var_so2                  5.833631
calc_var_no2                  5.833631
spectrogram_mean_coeff_co     5.833631
spectrogram_mean_coeff_no2    5.833631
calc_var_co                   5.833631
wavelet_var_co                5.833631


## Kesimpulan 5.6 dan cara menjalankan ulang

- Matriks gabungan, hasil PCA, dan seluruh grid tersimpan di `data/processed/`; gambar di `data/images/eksperimen204/`.
- Target dimensi 74/37 yang tidak layak akan otomatis aktif bila jumlah daerah bertambah (tambahkan baris ke `ekstraksi_fitur_*.csv`, jalankan ulang notebook ini).
- Label cluster terbaik (k dengan silhouette tertinggi) dipakai notebook 5.7 untuk peta Folium.